In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from cyclegan_core import denormalize, seed_everything
from two_stage_virtual_staining import (
    ColorizerTrainer, build_structure_trainer, build_two_stage_dataloaders
)

/home/user/anaconda3/envs/urban/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Two-stage virtual H&E staining at 2.0 MPP

Stage 1 uses a true 1-channel paired bidirectional CycleGAN to complete `Unstain OD ↔ H&E grayscale OD` without any blurred loss. Sharp Stage 2 receives only the Stage-1 predicted H&E OD, uses full-resolution SSIM, RGB gradient/Laplacian losses, and residual detail refinement to generate RGB H&E. The complete 2048×2048 patch at 0.5 MPP is resized to 512×512 at 2.0 MPP.

In [2]:
TRAIN_STAGE1 = False  # reuse the existing v2 Stage-1 best checkpoint

data_params = {
    'seed': 42,
    'gpu_index': 1,
    'data_dir': Path('../../data/HnE_n_UNStaining/patch_dataset_mpp05_2048'),
    'image_ext': 'png',
    'image_max_count': 30000,
    'original_size': 2048,
    'source_mpp': 0.5,
    'target_mpp': 2.0,
    'input_size': 512,
    'batch_size': 2,
    'val_fraction': 0.10,
    'preload_images': True,
    'max_cache_gib': 64,
    'od_background_threshold': 0.98,
    'od_quantile': 0.995,
    'od_calibration_images': 256,
}

structure_params = {
    **data_params,
    'output_dir': Path('../../results/Unstain2HnE_two_stage_v2/structure'),
    'checkpoint_dir': Path('../../model/Unstain2HnE_two_stage_v2/structure'),
    'num_epochs': 100,
    'decay_start_epoch': 50,
    'ngf': 32,
    'ndf': 32,
    'residual_blocks': 6,
    'lr_g': 2e-4,
    'lr_d': 1e-4,
    'beta1': 0.5,
    'beta2': 0.999,
    'lambda_gan': 1.0,
    'lambda_cycle': 5.0,
    'lambda_identity': 2.5,
    'lambda_background': 5.0,
    'lambda_paired': 20.0,
    'lambda_ssim': 2.0,
    'lambda_gradient': 1.0,
    'a_background_od_threshold': 0.04,
    'b_background_od_threshold': 0.04,
    'background_mask_blur_kernel': 1,  # no Stage-1 blur; hard OD background mask
    'pool_size': 50,
    'preview_count': 2,
    'save_every': 10,
}

color_params = {
    'output_dir': Path('../../results/Unstain2HnE_two_stage_v4/color'),
    'checkpoint_dir': Path('../../model/Unstain2HnE_two_stage_v4/color'),
    'num_epochs': 100,
    'base_channels': 32,
    'input_channels': 1,  # Stage-1 predicted H&E OD only
    'detail_refinement': True,
    'ndf': 64,
    'lr_g': 2e-4,
    'lr_d': 1e-4,
    'beta1': 0.5,
    'beta2': 0.999,
    'lambda_gan': 1.0,
    'lambda_rgb': 10.0,
    'lambda_ssim': 2.0,
    'lambda_gradient': 2.0,
    'lambda_laplacian': 1.0,
    'lambda_background': 10.0,
    'background_od_threshold': 0.04,
    'background_mask_blur_kernel': 1,
    'preview_count': 2,
    'save_every': 10,
}

seed_everything(data_params['seed'])
if torch.cuda.is_available():
    device = torch.device(f"cuda:{data_params['gpu_index']}")
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device('cpu')
print('device:', device)

device: cuda:1


## DataLoader

Unstain and H&E use the same filename, full field of view, and spatial augmentation. Separate fixed global OD maxima are calibrated at the actual 512×512 training scale; no per-image min-max or z-score normalization is used.

In [ ]:
data = build_two_stage_dataloaders(data_params)
print('Unstain OD_MAX:', data['unstain_od_max'])
print('H&E OD_MAX:', data['hne_od_max'])

Preloading RGB originals into RAM:   3%|▎         | 54/2104 [00:05<03:43,  9.17it/s]

In [ ]:
unstain_od, hne_rgb = next(iter(data['color_train']))
columns = min(4, len(unstain_od))
fig, axes = plt.subplots(2, columns, figsize=(4 * columns, 7), squeeze=False)
for i in range(columns):
    axes[0, i].imshow(denormalize(unstain_od[i, 0]).numpy(), cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title('Unstain OD')
    axes[1, i].imshow(denormalize(hne_rgb[i]).permute(1, 2, 0).numpy())
    axes[1, i].set_title('Target RGB H&E')
    for row in range(2):
        axes[row, i].axis('off')
plt.tight_layout()

## Stage 1 — paired CycleGAN structure completion

Both directions remain active: `Unstain OD → H&E OD → Unstain OD` and `H&E OD → Unstain OD → H&E OD`. Stage 1 is strictly 1-channel, uses no blur anywhere, lowers cycle pressure, and learns from direct paired L1, SSIM, OD-gradient, adversarial, identity, and background losses. Best-checkpoint selection uses only forward H&E-OD fidelity.

In [ ]:
structure_trainer = build_structure_trainer(structure_params, data, device)

In [ ]:
if TRAIN_STAGE1:
    structure_trainer.fit()
else:
    print('Skipping Stage 1 training; the existing best checkpoint will be loaded next.')

In [ ]:
best_structure_path = structure_params['checkpoint_dir'] / 'best.pt'
best_structure = torch.load(
    best_structure_path, map_location=device, weights_only=False
)
structure_trainer.G_AB.load_state_dict(best_structure['G_AB'])
structure_trainer.G_BA.load_state_dict(best_structure['G_BA'])
structure_trainer.G_AB.eval()
print('Loaded best structure epoch:', best_structure['epoch'] + 1)

## Stage 2 — sharp conditional RGB H&E colorization

The colorizer always receives only `Stage-1 predicted H&E OD`. Original Unstain OD concatenation, real-H&E-OD teacher forcing, and the oracle colorization branch are removed. It uses 512×512 full-resolution SSIM, direct RGB gradient and Laplacian losses, residual detail refinement after bilinear upsampling, and a hard background mask (`kernel=1`). Stage 1 remains frozen, while real H&E RGB is used only as the Stage-2 output target. Existing Color checkpoints are intentionally not reused.

In [ ]:
color_trainer = ColorizerTrainer(
    color_params, data['color_train'], data['color_val'],
    structure_trainer.G_AB, device
)

In [ ]:
color_trainer.fit()

## Full-pipeline validation

All metrics evaluate only the deployed path: `Unstain OD → predicted H&E OD → predicted RGB H&E`. Real H&E OD is never passed to Stage 2; real H&E RGB is used only as the final target.

In [ ]:
metrics, preview = color_trainer.validate()
print(metrics)
color_trainer.save_preview(max(color_trainer.start_epoch - 1, 0), preview)